In [1]:
# import os
# import pandas as pd

# folder = "raw/job"
# all_dfs = []

# # Lặp qua tất cả các file trong thư mục
# for filename in os.listdir(folder):
#     if filename.endswith(".csv"):
#         file_path = os.path.join(folder, filename)
#         # Đọc file và thêm vào danh sách
#         df = pd.read_csv(file_path)
#         all_dfs.append(df)

# # Nối tất cả các dataframe lại với nhau
# combined_df = pd.concat(all_dfs, ignore_index=True)

# # Lưu thành file mới để tiền xử lý
# combined_df.to_csv("combined_data.csv", index=False, encoding='utf-8-sig')

# print(f"Đã gộp xong! Tổng số dòng: {len(combined_df)}")



In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib as plt
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
df=pd.read_csv("raw/job/combined_data.csv", encoding='utf-8-sig')

In [4]:
import pandas as pd
import re
import numpy as np

def final_clean_salary_row(row):
    raw_min = str(row['salary_min']).lower().strip() if pd.notna(row['salary_min']) else ""
    raw_max = str(row['salary_max']).lower().strip() if pd.notna(row['salary_max']) else ""
    combined_text = raw_min + " " + raw_max

    # 1. Type 3: Thỏa thuận
    keywords = ['thỏa thuận', 'cạnh tranh', 'thương lượng', 'negotiable']
    if any(k in combined_text for k in keywords) or (raw_min == "" and raw_max == ""):
        return pd.Series([np.nan, np.nan, 3])

    # 2. Lấy toàn bộ số
    def get_nums(text):
        clean_str = text.replace('.', '').replace(',', '')
        return [float(n) for n in re.findall(r'\d+', clean_str)]

    all_nums = get_nums(combined_text)
    if not all_nums:
        return pd.Series([np.nan, np.nan, 3])

    is_usd = 'usd' in combined_text

    def calc(val):
        if is_usd:
            val *= 26335
        # Nhân 1 triệu nếu số nhỏ hoặc có từ "tr"/"triệu"
        if val < 1000 or 'triệu' in combined_text or re.search(r'\btr\b', combined_text):
            return val * 1_000_000
        return val

    # 3. Dùng \b để tránh false positive với 'từ'
    is_under = bool(re.search(r'\b(dưới|đến|tới|lên đến|upto|up to)\b', combined_text))
    is_over  = bool(re.search(r'\b(trên|hơn|tối thiểu|từ|from)\b', combined_text))

    # ⚡ Xử lý đặc biệt: "Trên X triệu" (ví dụ: "Trên 40 triệu")
    # Ưu tiên kiểm tra combined_text của từng ô riêng
    raw_over  = bool(re.search(r'\b(trên|hơn|tối thiểu|từ|from)\b', raw_min + " " + raw_max))
    raw_under = bool(re.search(r'\b(dưới|đến|tới|lên đến|upto|up to)\b', raw_min + " " + raw_max))

    f_min, f_max = np.nan, np.nan
    s_type = 0

    if len(all_nums) >= 2:
        n1, n2 = calc(all_nums[0]), calc(all_nums[1])
        f_min, f_max, s_type = min(n1, n2), max(n1, n2), 0

    elif len(all_nums) == 1:
        val = calc(all_nums[0])
        if raw_over and not raw_under:
            # "Trên 40 triệu" → min=40tr, max=NaN, type=1
            f_min, f_max, s_type = val, np.nan, 1
        elif raw_under and not raw_over:
            # "Dưới 20 triệu" → min=NaN, max=20tr, type=2
            f_min, f_max, s_type = np.nan, val, 2
        else:
            # Chỉ 1 số, không rõ hướng → coi là mức sàn
            f_min, f_max, s_type = val, np.nan, 1

    return pd.Series([f_min, f_max, s_type])

In [5]:
def clean_and_impute_salary(df):
    df['main_industry'] = df['industries'].str.split(',').str[0].str.strip()

    # --- BƯỚC 1: XỬ LÝ THEO LOẠI LƯƠNG (Ưu tiên dữ liệu gốc) ---
    # Nếu Type 1 (Trên 40tr) và Max đang trống -> Ép Max = Min * 1.2
    mask_over = (df['salary_type'] == 1) & (df['salary_max'].isna())
    df.loc[mask_over, 'salary_max'] = df.loc[mask_over, 'salary_min'] * 1.2

    # Nếu Type 2 (Dưới 20tr) và Min đang trống -> Ép Min = Max * 0.8
    mask_under = (df['salary_type'] == 2) & (df['salary_min'].isna())
    df.loc[mask_under, 'salary_min'] = df.loc[mask_under, 'salary_max'] * 0.8

    # --- BƯỚC 2: IMPUTATION THEO TẦNG (Cho những ô còn NaN - Type 3 hoặc thiếu cả cặp) ---
    levels = [
        ['main_industry', 'job_experience_required', 'employment_type'],
        ['main_industry', 'job_experience_required'],
        ['main_industry'],
        ['employment_type']
    ]

    for cols in levels:
        df['salary_min'] = df['salary_min'].fillna(df.groupby(cols)['salary_min'].transform('median'))
        df['salary_max'] = df['salary_max'].fillna(df.groupby(cols)['salary_max'].transform('median'))
    
    # --- BƯỚC 3: ĐIỀN GLOBAL MEDIAN CHO CÁC Ô TRỐNG HOÀN TOÀN ---
    df['salary_min'] = df['salary_min'].fillna(df['salary_min'].median())
    df['salary_max'] = df['salary_max'].fillna(df['salary_max'].median())
    
    # Trường hợp hy hữu nếu vẫn thiếu 1 đầu sau khi điền median
    df['salary_max'] = df['salary_max'].fillna(df['salary_min'] * 1.2)
    df['salary_min'] = df['salary_min'].fillna(df['salary_max'] * 0.8)

    # --- BƯỚC 4: CHỐT CHẶN CUỐI CÙNG (Đảm bảo Min < Max cho mô hình) ---
    # Nếu do median điền vào làm Min > Max, ta điều chỉnh Max lên thay vì swap
    mask_error = df['salary_min'] >= df['salary_max']
    df.loc[mask_error, 'salary_max'] = df.loc[mask_error, 'salary_min'] * 1.2

    # --- BƯỚC 5: ÉP KIỂU ---
    df['salary_min'] = df['salary_min'].round().astype('int64')
    df['salary_max'] = df['salary_max'].round().astype('int64')
    df['salary_type'] = df['salary_type'].astype('int')
    
    return df

In [6]:
df[['salary_min', 'salary_max', 'salary_type']] = df.apply(final_clean_salary_row, axis=1)

df = clean_and_impute_salary(df)


In [7]:
import re
import nltk
from textblob import TextBlob
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from textblob import Word
from nltk.util import ngrams
import re
from wordcloud import WordCloud, STOPWORDS
from nltk.tokenize import word_tokenize

In [8]:
import re
import pandas as pd

def clean_experience(text):
    # Kiểm tra nếu text là NaN (float) hoặc không phải là chuỗi
    if pd.isna(text) or not isinstance(text, str):
        return 0, 0  # Hoặc giá trị mặc định bạn muốn
    
    # Xử lý trường hợp "Lên đến"
    if "Lên đến" in text:
        nums = re.findall(r'\d+', text)
        return 0, (int(nums[0]) if nums else 1)
    
    # Tìm tất cả các số
    nums = re.findall(r'\d+', text)
    
    if len(nums) >= 2: 
        return int(nums[0]), int(nums[1])
    if len(nums) == 1:
        if "Trên" in text: 
            return int(nums[0]), 20 # Giả định trần 20 năm
        return int(nums[0]), int(nums[0])
        
    return 0, 0

In [9]:
df[["exp_min", "exp_max"]] = df["job_experience_required"].apply(
    lambda x: pd.Series(clean_experience(x))
)
df.drop(columns=["job_experience_required"], inplace=True)

In [10]:
# import re
# import pandas as pd


# with open("vietnamese-stopwords.txt", "r", encoding="utf-8") as f:
#     vietnamese_stopwords = set([line.strip() for line in f if line.strip()])

# import re
# import pandas as pd


# with open("vietnamese-stopwords.txt", "r", encoding="utf-8") as f:
#     vietnamese_stopwords = set([line.strip() for line in f if line.strip()])

# def advanced_clean_text(text):
#     if pd.isna(text): return ""
#     text = str(text).lower()
    
#     # 1. Bảo vệ từ khóa kỹ thuật (C#, .NET...)
#     text = re.sub(r'\bc#\b', 'csharp', text)
#     text = re.sub(r'\.net\b', 'dotnet', text)
    
#     # 2. Tách dấu chấm THÔNG MINH:
#     # Chỉ tách dấu chấm nếu trước hoặc sau nó KHÔNG phải là chữ số
#     # Giúp bảo vệ 10.000.000 hoặc 3.5 năm
#     text = re.sub(r'(?<!\d)\.|\.(?!\d)', ' . ', text)
    
#     # 3. Loại bỏ các ký tự đặc biệt khác nhưng giữ lại chữ cái và số
#     text = re.sub(r'[^\w\s\.]', ' ', text)
    
#     # 4. Xóa các từ dừng
#     words = text.split()
#     words = [w for w in words if w not in vietnamese_stopwords and len(w) > 1 or w == '.']
    
#     return " ".join(words).strip()

# job_specific_stopwords = {
#     'mô', 'tả', 'công', 'việc', 'yêu', 'cầu', 'quyền', 'lợi', 'thực', 'hiện', 
#     'chức', 'danh', 'vị', 'trí', 'trách', 'nhiệm', 'trao', 'đổi', 'chi', 'tiết', 
#     'phỏng', 'vấn', 'hồ', 'sơ', 'ứng', 'tuyển', 'tốt', 'nghiệp', 'kinh', 'nghiệm'
# }
# vietnamese_stopwords.update(job_specific_stopwords)

# # Áp dụng
# df['job_description'] = df['job_description'].apply(advanced_clean_text)
# df['job_requirement'] = df['job_requirement'].apply(advanced_clean_text)

# # Ghép text để làm đầu vào cho TF-IDF (như code trước của bạn)
# df["job_text"] = df["job_description"] + " " + df["job_requirement"]


In [11]:
import pandas as pd

# 1. Loại outlier salary
# Lưu ý: 500_000_000 là 500 triệu. Nếu là 500 tỷ phải là 500_000_000_000
df = df[df['salary_max'] < 500_000_000] 

# 2. Chuẩn hóa employment_type → Nhóm lại thành các type sạch
def normalize_emp(text):
    if pd.isna(text): 
        return 'full_time'
    
    t = str(text).lower()
    
    # THỨ TỰ ƯU TIÊN: 
    # Check các loại hình đặc thù trước, nếu không khớp mới về mặc định full_time
    
    # 1. Thực tập (Intern)
    if 'thực tập' in t or 'intern' in t: 
        return 'intern'
    
    # 2. Bán thời gian (Part-time)
    if 'bán thời gian' in t or 'part time' in t or 'part-time' in t: 
        return 'part_time'
    
    # 3. Thời vụ / Freelance / Khác
    if any(k in t for k in ['thời vụ', 'tự do', 'freelance', 'others']): 
        return 'freelance'
    
    # 4. Remote (Làm từ xa)
    if 'remote' in t or 'từ xa' in t: 
        return 'remote'
    
    # 5. Mặc định: Nhân viên chính thức / Full time
    # Bao gồm cả các trường hợp nhiễu như "Nhân viên chính thức, Hình thức tăng lương..."
    if 'nhân viên chính thức' in t or 'full time' in t or 'fulltime' in t:
        return 'full_time'
    
    return 'full_time'

# Áp dụng chuẩn hóa
df['employment_type'] = df['employment_type'].apply(normalize_emp)

# 3. Dedup industries (Dọn dẹp ngành nghề)
df['industries'] = df['industries'].apply(
    lambda x: list(dict.fromkeys([i.strip() for i in str(x).split(',')])) if pd.notna(x) else []
)

# Kiểm tra lại kết quả
print(df['employment_type'].value_counts())

employment_type
full_time    36833
part_time     1007
freelance      565
intern         565
remote          66
Name: count, dtype: int64


In [12]:
import pandas as pd
import numpy as np

def map_industry_group(main_industry):
    if pd.isna(main_industry):
        return 'Khác'
    
    text = str(main_industry).lower()
    
    # 1. Công nghệ thông tin
    if any(k in text for k in ['it', 'cntt', 'phần mềm', 'phần cứng', 'lập trình', 'developer', 'mạng', 'tester', 'seo']):
        return 'Công nghệ thông tin'
    
    # 2. Tài chính - Kế toán
    if any(k in text for k in ['kế toán', 'kiểm toán', 'tài chính', 'ngân hàng', 'chứng khoán', 'bảo hiểm', 'đầu tư', 'tín dụng', 'thẩm định']):
        return 'Tài chính - Kế toán'
    
    # 3. Kinh doanh - Bán hàng
    if any(k in text for k in ['bán hàng', 'kinh doanh', 'sale', 'telesale', 'thương mại điện tử', 'bán lẻ', 'bán sỉ', 'phát triển thị trường', 'showroom']):
        return 'Kinh doanh - Bán hàng'
    
    # 4. Marketing - Truyền thông
    if any(k in text for k in ['marketing', 'tiếp thị', 'quảng cáo', 'truyền thông', 'đối ngoại', 'pr', 'copywriter', 'content', 'biên tập', 'sự kiện']):
        return 'Marketing - Truyền thông'
    
    # 5. Kỹ thuật - Sản xuất
    if any(k in text for k in ['kỹ thuật', 'cơ khí', 'ô tô', 'điện', 'điện tử', 'tự động hóa', 'sản xuất', 'vận hành', 'bảo trì', 'sửa chữa', 'qa', 'qc', 'an toàn lao động']):
        return 'Kỹ thuật - Sản xuất'
    
    # 6. Xây dựng - Kiến trúc - BĐS
    if any(k in text for k in ['xây dựng', 'kiến trúc', 'nội thất', 'ngoại thất', 'bất động sản', 'nhà đất', 'địa chính', 'trắc địa', 'cầu đường']):
        return 'Xây dựng - BĐS'
    
    # 7. Dịch vụ - Du lịch - F&B
    if any(k in text for k in ['khách sạn', 'nhà hàng', 'du lịch', 'thực phẩm', 'đồ uống', 'f&b', 'pha chế', 'đầu bếp', 'phục vụ', 'spa', 'làm đẹp', 'thẩm mỹ']):
        return 'Dịch vụ - F&B - Làm đẹp'
    
    # 8. Vận tải - Logistics
    if any(k in text for k in ['vận tải', 'vận chuyển', 'logistics', 'kho vận', 'giao nhận', 'xuất nhập khẩu', 'lái xe', 'phụ xe', 'hàng hải', 'hàng không']):
        return 'Vận tải - Logistics'
    
    # 9. Y tế - Dược - Sinh học
    if any(k in text for k in ['y tế', 'dược', 'bác sĩ', 'y tá', 'điều dưỡng', 'sinh học', 'hóa học', 'mỹ phẩm']):
        return 'Y tế - Dược'
    
    # 10. Hành chính - Nhân sự - Pháp lý
    if any(k in text for k in ['nhân sự', 'hành chính', 'văn phòng', 'thư ký', 'trợ lý', 'luật', 'pháp lý', 'biên phiên dịch', 'lễ tân', 'tổng vụ']):
        return 'Hành chính - Nhân sự'
    
    # 11. Giáo dục - Đào tạo
    if any(k in text for k in ['giáo dục', 'đào tạo', 'giảng viên', 'giáo viên', 'trợ giảng', 'học vụ']):
        return 'Giáo dục - Đào tạo'
    
    # 12. Lao động phổ thông
    if any(k in text for k in ['công nhân', 'lao động phổ thông', 'giúp việc', 'tạp vụ', 'bảo vệ', 'an ninh', 'giao hàng']):
        return 'Lao động phổ thông'
    
    # 13. Nông - Lâm - Ngư nghiệp
    if any(k in text for k in ['nông nghiệp', 'lâm nghiệp', 'thủy sản', 'hải sản', 'chăn nuôi', 'thú y', 'trồng trọt']):
        return 'Nông - Lâm - Ngư nghiệp'

    return 'Khác'

# Áp dụng map
df['industry_group'] = df['main_industry'].apply(map_industry_group)

# Kiểm tra phân bổ
print(df['industry_group'].value_counts())

industry_group
Kinh doanh - Bán hàng       6729
Tài chính - Kế toán         6556
Khác                        5721
Kỹ thuật - Sản xuất         4123
Hành chính - Nhân sự        3162
Xây dựng - BĐS              2896
Marketing - Truyền thông    2881
Công nghệ thông tin         1717
Dịch vụ - F&B - Làm đẹp     1632
Vận tải - Logistics         1492
Giáo dục - Đào tạo           892
Y tế - Dược                  761
Lao động phổ thông           347
Nông - Lâm - Ngư nghiệp      127
Name: count, dtype: int64


In [13]:
import pandas as pd
import numpy as np

# 1. Danh sách 63 tỉnh thành (Dạng List)
provinces_list = [
    'Hà Nội', 'Hồ Chí Minh', 'Đà Nẵng', 'Hải Phòng', 'Cần Thơ',
    'An Giang', 'Bà Rịa - Vũng Tàu', 'Bắc Giang', 'Bắc Kạn', 'Bạc Liêu',
    'Bắc Ninh', 'Bến Tre', 'Bình Định', 'Bình Dương', 'Bình Phước',
    'Bình Thuận', 'Cà Mau', 'Cao Bằng', 'Đắk Lắk', 'Đắk Nông',
    'Điện Biên', 'Đồng Nai', 'Đồng Tháp', 'Gia Lai', 'Hà Giang',
    'Hà Nam', 'Hà Tĩnh', 'Hải Dương', 'Hậu Giang', 'Hòa Bình',
    'Hưng Yên', 'Khánh Hòa', 'Kiên Giang', 'Kon Tum', 'Lai Châu',
    'Lâm Đồng', 'Lạng Sơn', 'Lào Cai', 'Long An', 'Nam Định',
    'Nghệ An', 'Ninh Bình', 'Ninh Thuận', 'Phú Thọ', 'Quảng Bình',
    'Quảng Nam', 'Quảng Ngãi', 'Quảng Ninh', 'Quảng Trị', 'Sóc Trăng',
    'Sơn La', 'Tây Ninh', 'Thái Bình', 'Thái Nguyên', 'Thanh Hóa',
    'Thừa Thiên Huế', 'Tiền Giang', 'Trà Vinh', 'Tuyên Quang', 'Vĩnh Long',
    'Vĩnh Phúc', 'Yên Bái', 'Phú Yên'
]

# 2. Bảng map các tên viết tắt (Dạng Dictionary)
province_aliases = {
    'TP.HCM': 'Hồ Chí Minh',
    'TP HCM': 'Hồ Chí Minh',
    'Sài Gòn': 'Hồ Chí Minh',
    'HCMC': 'Hồ Chí Minh',
    'Vũng Tàu': 'Bà Rịa - Vũng Tàu',
    'Huế': 'Thừa Thiên Huế',
    'Dak Lak': 'Đắk Lắk',
}

def extract_province(addr):
    if pd.isna(addr): return 'Khác'
    
    addr_str = str(addr).lower()
    
    # Bước 1: Kiểm tra Alias (Dùng .items() cho Dictionary)
    for alias, formal in province_aliases.items():
        if alias.lower() in addr_str:
            return formal
            
    # Bước 2: Kiểm tra 63 tỉnh (Dùng vòng lặp cho List)
    for p in provinces_list:
        if p.lower() in addr_str:
            return p
            
    # Bước 3: Các trường hợp đặc thù trong tuyển dụng
    if any(k in addr_str for k in ['toàn quốc', 'tất cả', 'linh hoạt']):
        return 'Toàn quốc'
    if any(k in addr_str for k in ['nước ngoài', 'singapore', 'japan', 'usa', 'nhật bản']):
        return 'Nước ngoài'
    if any(k in addr_str for k in ['tại nhà', 'làm việc từ xa', 'remote']):
        return 'Tại Nhà'
        
    return 'Khác'

# Áp dụng
df['province'] = df['job_address'].apply(extract_province)

# Kiểm tra thử kết quả
print(df['province'].value_counts())

province
Hồ Chí Minh    13963
Hà Nội         12223
Khác            2751
Bình Dương      1457
Đồng Nai         868
               ...  
Phú Yên           14
Hậu Giang         13
Cao Bằng          10
Đắk Nông           8
Bắc Kạn            5
Name: count, Length: 66, dtype: int64


In [14]:
# df.drop(columns=["main_industry","industries","job_description", "job_requirement", "job_address"], inplace=True)

In [15]:
# import re
# import pandas as pd
# from underthesea import word_tokenize # Thư viện tách từ tiếng Việt
# from sklearn.preprocessing import MinMaxScaler

# # 1. Bảo vệ kỹ từ khóa kỹ thuật và tách từ
# def gnn_clean_text(text):
#     if pd.isna(text): return ""
#     text = str(text).lower()
    
#     # Bảo vệ C#, C++, .NET trước khi xóa ký tự đặc biệt
#     text = re.sub(r'\bc#\b', 'csharp', text)
#     text = re.sub(r'\bc\+\+\b', 'cpp', text)
#     text = re.sub(r'\.net\b', 'dotnet', text)
    
#     # Xóa ký tự đặc biệt nhưng giữ lại dấu chấm cho các từ như node.js
#     # Và giữ lại dấu gạch dưới để phục vụ tách từ sau này
#     text = re.sub(r'[^\w\s\.]', ' ', text)
    
#     # Tách từ tiếng Việt (VD: "nhân viên" -> "nhân_viên")
#     text = word_tokenize(text, format="text")
    
#     # Lọc stopwords (sử dụng list bạn đã có)
#     words = text.split()
#     words = [w for w in words if w not in vietnamese_stopwords and len(w) > 1]
    
#     return " ".join(words)

# # 2. Chuẩn hóa dữ liệu số (Min-Max Scaling)
# # Đây là bước quan trọng để tạo Node Feature Matrix X
# from sklearn.preprocessing import RobustScaler

# scaler = RobustScaler()
# numerical_cols = ['salary_min', 'salary_max', 'exp_min', 'exp_max']

# df[numerical_cols] = scaler.fit_transform(df[numerical_cols].fillna(0))

# # 3. Mã hóa biến phân loại (One-Hot Encoding)
# # Chuyển province, industry_group thành các vector số
# df_encoded = pd.get_dummies(df, columns=['province', 'industry_group', 'employment_type'])



In [16]:
df.to_csv("processed/COMBINED_DATA_PROCESSED1.csv", index=False, encoding='utf-8-sig')

In [17]:
# import pandas as pd
# import numpy as np
# import re
# from sklearn.feature_extraction.text import TfidfVectorizer

# df_job  = pd.read_csv("combined_data7.csv")
# df_user = pd.read_csv("USER_DATA_FINAL.csv")

# # ── Bước 1: Tiền xử lý text (đã có từ pipeline trước) ──────────────
# # job_description và job_requirement đã được clean_text_keep_dot()
# # Ghép lại thành 1 chuỗi đại diện cho mỗi job
# df_job["job_text"] = (
#     df_job["job_description"].fillna("") + " " +
#     df_job["job_requirement"].fillna("")
# )

# # User text: Skills + Target
# df_user["user_text"] = (
#     df_user["Skills"].fillna("") + " " +
#     df_user["Target"].fillna("")
# )

# # ── Bước 2: Fit TF-IDF trên TOÀN BỘ corpus (job + user) ────────────
# # Quan trọng: phải fit chung để vocabulary nhất quán
# all_texts = pd.concat([
#     df_job["job_text"],
#     df_user["user_text"]
# ], ignore_index=True)

# tfidf = TfidfVectorizer(
#     max_features=256,        # đủ dùng cho GNN, nhẹ hơn bge-m3 4x
#     ngram_range=(1, 2),      # unigram + bigram để giữ ngữ cảnh cụm từ
#     sublinear_tf=True,       # log-scaling, giảm ảnh hưởng từ xuất hiện nhiều
#     min_df=3,                # bỏ từ cực hiếm (< 3 docs)
#     max_df=0.9,              # bỏ từ quá phổ biến (> 90% docs)
# )

# tfidf.fit(all_texts)

# # ── Bước 3: Transform riêng từng tập ────────────────────────────────
# job_text_emb  = tfidf.transform(df_job["job_text"]).toarray()   # (39038, 256)
# user_text_emb = tfidf.transform(df_user["user_text"]).toarray() # (3983,  256)

# print(f"Job text embedding:  {job_text_emb.shape}")
# print(f"User text embedding: {user_text_emb.shape}")

# # ── Bước 4: Gắn vector vào dataframe (cách đúng) ────────────────────
# # KHÔNG dùng enumerate(df[col1], df[col2])
# # Chuyển thẳng thành cột hoặc lưu numpy array riêng
# # Option A: lưu numpy array (khuyến nghị cho PyG)
# np.save("job_text_emb.npy",  job_text_emb)
# np.save("user_text_emb.npy", user_text_emb)

# # Option B: nếu muốn gắn vào df (chỉ dùng để debug)
# df_job["text_vector"]  = list(job_text_emb)
# df_user["text_vector"] = list(user_text_emb)

In [18]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer


# ── Bước 2: Fit TF-IDF trên TOÀN BỘ corpus (job + user) ────────────
# Quan trọng: phải fit chung để vocabulary nhất quán
all_texts = pd.concat([
    df["job_text"],
], ignore_index=True)

tfidf = TfidfVectorizer(
    max_features=256,        # đủ dùng cho GNN, nhẹ hơn bge-m3 4x
    ngram_range=(1, 2),      # unigram + bigram để giữ ngữ cảnh cụm từ
    sublinear_tf=True,       # log-scaling, giảm ảnh hưởng từ xuất hiện nhiều
    min_df=3,                # bỏ từ cực hiếm (< 3 docs)
    max_df=0.9,              # bỏ từ quá phổ biến (> 90% docs)
)

tfidf.fit(all_texts)

# ── Bước 3: Transform riêng từng tập ────────────────────────────────
job_text_emb  = tfidf.transform(df["job_text"]).toarray()   # (39038, 256)

print(f"Job text embedding:  {job_text_emb.shape}")

# ── Bước 4: Gắn vector vào dataframe (cách đúng) ────────────────────
# KHÔNG dùng enumerate(df[col1], df[col2])
# Chuyển thẳng thành cột hoặc lưu numpy array riêng
# Option A: lưu numpy array (khuyến nghị cho PyG)
np.save("job_text_emb.npy",  job_text_emb)

# Option B: nếu muốn gắn vào df (chỉ dùng để debug)
df["text_vector"]  = list(job_text_emb)


KeyError: 'job_text'